In [1]:
import sys
from pathlib import Path

import os
import requests
from dotenv import load_dotenv

import warnings
# on va ignorer les messages avertissements a cause d'un RAGASS pas compatible mystral v2.0
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

load_dotenv()

TOKEN = os.getenv("API_KEY")
TOKEN_MISTRAL = os.getenv("MISTRAL_API_KEY")

API_URL_ASK = "http://127.0.0.1:8000/ask"
API_URL_ASK_DEBUG = "http://127.0.0.1:8000/ask/debug"


In [3]:
#appel de mon api (probleme de versionnage de mistral)
question = "Quels événements culturels gratuits ont lieu à Montpellier ?"

resp = requests.post(
    API_URL_ASK,
    json={"question": question},
    headers={"x-api-key": TOKEN},
    timeout=60,
)

print(resp.status_code)
print(resp.json())

200
{'question': 'Quels événements culturels gratuits ont lieu à Montpellier ?', 'answer': "Je ne trouve pas d'événement correspondant, je suis un assistant qui ne peut vous conseiller que des événements culturels.", 'n_docs': 3, 'documents': [{'title': 'Visite libre de la bibliothèque Jean Claparède', 'location_name': 'Musée Fabre', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2025-09-20', 'last_date': '2025-09-21', 'event_type': '', 'url': 'https://openagenda.com/jep-2025-occitanie/events/bibliotheque-jean-claparede', 'score': None}, {'title': 'Conférences, expositions et dédicaces de livres au Cercle Culturel Languedocien !', 'location_name': 'Cercle Culturel Languedocien', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2025-09-20', 'last_date': '2025-09-21', 'event_type': '', 'url': 'https://openagenda.com/jep-2025-occitanie/events/visite-du-cercle-culturel-languedocien-4339688', 'score': None}, {'title': 'Exposition "Montpellier, regarder la ville aut

In [4]:
# question
question = "Quels événements culturels gratuits ont lieu à Montpellier ?"

resp = requests.post(
    API_URL_ASK,
    json={"question": question},
    headers={"x-api-key": TOKEN},
    timeout=60,
)

print(resp.status_code)
data = resp.json()
print(data["answer"])

200
Je ne trouve pas d'événement correspondant, je suis un assistant qui ne peut vous conseiller que des événements culturels.


In [5]:
import requests

def run_rag_for_eval(question: str) -> dict:
    resp = requests.post(
        API_URL_ASK_DEBUG,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS:", resp.status_code)
    print("TEXT:", resp.text)

    try:
        data = resp.json()
    except Exception:
        data = {"raw_text": resp.text}

    if resp.status_code != 200:
        return {
            "question": question,
            "error": data,
            "answer": None,
            "contexts": None,
        }

    return {
        "question": question,
        "answer": data["answer"],
        "contexts": data["contexts"],
    }

In [6]:
eval_questions = [

{
    "question": "Y a-t-il des expositions à Montpellier ?",
    "ground_truth": (
        "À Montpellier, les expositions présentes dans le corpus sont : "
        "L'association Miss'terre fait son expo d'été les 27 et 28 juin 2025 ; "
        "Exposition 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "aux Archives de Montpellier le 20 septembre 2025 ; "
        "Exposition 'L'Europe et son patrimoine' à la Maison de l'Europe "
        "de Montpellier les 20 et 21 septembre 2025."
    )
},

{
    "question": "Quels événements ont lieu au Musée Fabre ?",
    "ground_truth": (
        "Au Musée Fabre à Montpellier, un événement présent dans le corpus est : "
        "la visite libre de la bibliothèque Jean Claparède les 20 et 21 septembre 2025, "
        "avec une lecture autour de l'architecture et de la poésie occitane."
    )
},

{
    "question": "Quels événements ont lieu à Montpellier en septembre 2025 ?",
    "ground_truth": (
        "À Montpellier en septembre 2025, les événements présents dans le corpus sont : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre les 20 et 21 septembre ; "
        "les conférences, expositions et dédicaces au Cercle Culturel Languedocien les 20 et 21 septembre ; "
        "l'exposition 'Montpellier, regarder la ville autrement' aux Archives de Montpellier le 20 septembre ; "
        "et l'exposition 'L'Europe et son patrimoine' les 20 et 21 septembre."
    )
},

{
    "question": "Quels événements ont lieu aux Archives de Montpellier ?",
    "ground_truth": (
        "Aux Archives de Montpellier, l'événement présent dans le corpus est : "
        "l'exposition 'Montpellier, regarder la ville autrement. Archives et architecture', "
        "le 20 septembre 2025."
    )
},

{
    "question": "Quels événements culturels sont proposés à Montpellier ?",
    "ground_truth": (
        "Plusieurs événements culturels sont proposés à Montpellier dans le corpus : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
        "des conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
        "l'exposition 'Montpellier, regarder la ville autrement' aux Archives ; "
        "et l'exposition 'L'Europe et son patrimoine'."
    )
},

{
    "question": "Quels événements ont lieu le 20 septembre 2025 à Montpellier ?",
    "ground_truth": (
        "Le 20 septembre 2025 à Montpellier, les événements présents dans le corpus sont : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
        "les conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
        "et l'exposition 'Montpellier, regarder la ville autrement' aux Archives de Montpellier."
    )
},

{
    "question": "Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?",
    "ground_truth": (
        "Le week-end du 20 et 21 septembre 2025 à Montpellier comprend : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
        "les conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
        "et l'exposition 'L'Europe et son patrimoine'."
    )
},

{
    "question": "Y a-t-il des concerts de rock à Montpellier?",
    "ground_truth": (
        "Je ne trouve pas d'événement correspondant, je suis un assistant qui "
        "ne peut vous conseiller que des événements culturels."
    )
},

{
    "question": "Quels événements ont lieu à Paris ?",
    "ground_truth": (
        "Je ne trouve pas d'événement correspondant, je suis un assistant qui "
        "ne peut vous conseiller que des événements culturels."
    )
},

{
    "question": "Quels événements gratuits ont lieu à Montpellier ?",
    "ground_truth": (
        "Le corpus ne permet pas de confirmer explicitement la gratuité des événements, "
        "aucun événement ne peut donc être identifié avec certitude comme gratuit."
    )
}

]

In [7]:
rows = []

for item in eval_questions:
    result = run_rag_for_eval(item["question"])

    rows.append({
        "question": item["question"],
        "answer": result["answer"],
        "contexts": result["contexts"],
        "ground_truth": item["ground_truth"],
    })

rows[:1]

STATUS: 200
TEXT: {"question":"Y a-t-il des expositions à Montpellier ?","answer":"Titre : L'association Miss'terre fait son expo d'été !\n\nLieu : Association Miss'terre, Montpellier\nDate : 27 et 28 juin 2025\nDescription : Exposition organisée par l'association Miss'terre dans le cadre de son événement estival.\n\nPourquoi cet événement pourrait vous intéresser :\nSi vous aimez les expositions associatives et les événements locaux à Montpellier, cette expo d'été pourrait vous plaire, surtout si vous souhaitez découvrir des initiatives culturelles locales.\n\n---\n\nTitre : Exposition \"Montpellier, regarder la ville autrement. Archives et architecture\"\n\nLieu : Archives de Montpellier\nDate : 20 septembre 2025\nDescription : Présentation de documents issus des fonds des Archives de Montpellier illustrant l'évolution architecturale de cinq monuments emblématiques de la ville.\n\nPourquoi cet événement pourrait vous intéresser :\nSi vous êtes passionné par l'histoire, l'architecture

[{'question': 'Y a-t-il des expositions à Montpellier ?',
  'answer': 'Titre : L\'association Miss\'terre fait son expo d\'été !\n\nLieu : Association Miss\'terre, Montpellier\nDate : 27 et 28 juin 2025\nDescription : Exposition organisée par l\'association Miss\'terre dans le cadre de son événement estival.\n\nPourquoi cet événement pourrait vous intéresser :\nSi vous aimez les expositions associatives et les événements locaux à Montpellier, cette expo d\'été pourrait vous plaire, surtout si vous souhaitez découvrir des initiatives culturelles locales.\n\n---\n\nTitre : Exposition "Montpellier, regarder la ville autrement. Archives et architecture"\n\nLieu : Archives de Montpellier\nDate : 20 septembre 2025\nDescription : Présentation de documents issus des fonds des Archives de Montpellier illustrant l\'évolution architecturale de cinq monuments emblématiques de la ville.\n\nPourquoi cet événement pourrait vous intéresser :\nSi vous êtes passionné par l\'histoire, l\'architecture ou 

In [8]:
from ragas.dataset_schema import EvaluationDataset

ragas_rows = [
    {
        "user_input": row["question"],
        "response": row["answer"],
        "retrieved_contexts": row["contexts"],
        "reference": row["ground_truth"],
    }
    for row in rows[:3]   # commence petit pour éviter les 429
]

ragas_dataset = EvaluationDataset.from_list(ragas_rows)

print(ragas_dataset)
print(ragas_rows[0])

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=3)
{'user_input': 'Y a-t-il des expositions à Montpellier ?', 'response': 'Titre : L\'association Miss\'terre fait son expo d\'été !\n\nLieu : Association Miss\'terre, Montpellier\nDate : 27 et 28 juin 2025\nDescription : Exposition organisée par l\'association Miss\'terre dans le cadre de son événement estival.\n\nPourquoi cet événement pourrait vous intéresser :\nSi vous aimez les expositions associatives et les événements locaux à Montpellier, cette expo d\'été pourrait vous plaire, surtout si vous souhaitez découvrir des initiatives culturelles locales.\n\n---\n\nTitre : Exposition "Montpellier, regarder la ville autrement. Archives et architecture"\n\nLieu : Archives de Montpellier\nDate : 20 septembre 2025\nDescription : Présentation de documents issus des fonds des Archives de Montpellier illustrant l\'évolution architecturale de cinq monuments emblématiques de la ville.\n\nPourquoi cet

In [9]:
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset

llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=TOKEN_MISTRAL.strip(),
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
    column_map={
        "user_input": "question",
        "response": "answer",
        "retrieved_contexts": "contexts",
    },
)

print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Exception raised in Job[5]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')
Exception raised in Job[1]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')
Exception raised in Job[9]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')


{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_precision': 0.1111, 'context_recall': 0.5000}


In [10]:
result_df = result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,Y a-t-il des expositions à Montpellier ?,[L'association Miss'terre fait son expo d'été ...,Titre : L'association Miss'terre fait son expo...,"À Montpellier, les expositions présentes dans ...",1.0,NaN,0.000000,1.0
1,Quels événements ont lieu au Musée Fabre ?,"[Visite commentée : « Un musée, des bâtiments,...","Titre : Visite commentée : « Un musée, des bât...","Au Musée Fabre à Montpellier, un événement pré...",1.0,NaN,0.000000,0.0
2,Quels événements ont lieu à Montpellier en sep...,[Visite guidée du cimetière protestant de Mont...,Titre : Visite guidée du cimetière protestant ...,"À Montpellier en septembre 2025, les événement...",1.0,NaN,0.333333,0.5


In [11]:
print(result)

{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_precision': 0.1111, 'context_recall': 0.5000}
